# Credit Card Fraud Detection
By **Jamiu Dahunsi**

In this project, I explore how to handle extreme class imbalance using the popular Kaggle Credit Card Fraud dataset. 

Out of **284,807 transactions**, only **492 are fraudulent (~0.17%)**. 

A naive baseline predicting that *no transaction is ever fraud* would achieve **99.83% accuracy**, but it would be completely useless in real life. Here, we focus on what really matters:
- **Precision-Recall Area Under Curve (PR-AUC)**
- **Recall** (catching as much fraud as possible)
- **Precision** (avoiding annoying false alarms for legitimate customers)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve
)
import xgboost as xgb

sns.set_theme(style='whitegrid')
%matplotlib inline


## 1. Loading the Data & Sanity Check

In [ ]:
df = pd.read_csv('data/creditcard.csv')
print(f'Rows: {df.shape[0]:,}, Columns: {df.shape[1]}')
print('Missing values:', df.isnull().sum().sum())
df.head()


## 2. Looking at the Imbalance

In [ ]:
counts = df['Class'].value_counts()
print(f'Legitimate (0): {counts[0]:,} ({counts[0]/len(df):.3%})')
print(f'Fraudulent (1): {counts[1]:,} ({counts[1]/len(df):.3%})')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Legitimate (0)', 'Fraud (1)'], [counts[0], counts[1]], color=['#2b5c8f', '#d9534f'])
ax.set_yscale('log')
ax.set_title('Transaction Counts (Log Scale)')
plt.show()


## 3. Train / Test Split & Leak-Free Scaling
We use stratify=y to make sure both the train and test sets have the exact same 0.17% fraud prevalence. 
We also scale Amount and Time using RobustScaler, fitting only on the training set to avoid data leakage.


In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler_amount = RobustScaler()
scaler_time = RobustScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train['scaled_amount'] = scaler_amount.fit_transform(X_train[['Amount']])
X_train['scaled_time'] = scaler_time.fit_transform(X_train[['Time']])
X_train.drop(['Time', 'Amount'], axis=1, inplace=True)

X_test['scaled_amount'] = scaler_amount.transform(X_test[['Amount']])
X_test['scaled_time'] = scaler_time.transform(X_test[['Time']])
X_test.drop(['Time', 'Amount'], axis=1, inplace=True)

print(f'Train size: {len(X_train):,} ({y_train.sum()} frauds)')
print(f'Test size:  {len(X_test):,} ({y_test.sum()} frauds)')


## 4. Training Models: Baseline vs. Balanced Ensembles

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest (Balanced)': RandomForestClassifier(
        n_estimators=100, class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=150,
        learning_rate=0.1,
        max_depth=5,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric='aucpr',
        random_state=42,
        n_jobs=-1
    )
}

results = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    
    pr_auc = average_precision_score(y_test, y_prob)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    print(f'=== {name} ===')
    print(f'PR-AUC:  {pr_auc:.4f}')
    print(f'ROC-AUC: {roc_auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'], digits=4))


## 5. Visualizing Precision-Recall & Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, clf in models.items():
    y_prob = clf.predict_proba(X_test)[:, 1]
    p, r, _ = precision_recall_curve(y_test, y_prob)
    axes[0].plot(r, p, label=f'{name} (PR-AUC: {average_precision_score(y_test, y_prob):.3f})')

axes[0].set_title('Precision-Recall Curves')
axes[0].set_xlabel('Recall (Fraud Catch Rate)')
axes[0].set_ylabel('Precision (True Fraud / Flagged)')
axes[0].legend()

# Best model confusion matrix
y_pred_rf = models['Random Forest (Balanced)'].predict(X_test)
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar=False)
axes[1].set_title('Random Forest (Balanced) Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_xticklabels(['Legit', 'Fraud'])
axes[1].set_yticklabels(['Legit', 'Fraud'])

plt.tight_layout()
plt.show()


## 6. What I Learned
1. **Never evaluate imbalanced datasets on accuracy**: An accuracy of 99.8% can hide the fact that a model misses more than a third of the fraud cases.
2. **Precision vs. Recall in Business**:
   - High recall means catching almost all fraudulent transactions.
   - High precision means not declining legitimate customer transactions by mistake.
3. **The Best Trade-Off**:
   - **Balanced Random Forest** achieved a PR-AUC of **0.8610** with **96% precision** and only **3 false positives** across 56,864 genuine transactions.
   - **XGBoost with scale_pos_weight** gave the highest raw catch rate (**83.7% recall**) while maintaining a strong 73.9% precision.
